[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aalbata/ai-labs-for-high-school/blob/main/notebooks/object_recognition_yolov8.ipynb)

# Object Recognition with AI

Hands-On AI for High School, grades 9 to 12

Dr. Ali Al Bataineh, Higher Colleges of Technology, aalbataineh@hct.ac.ae

In this lab you use a pre-trained AI model called YOLOv8 to find and label objects in photos. YOLO stands for "You Only Look Once", the name of the method introduced in 2016 by the researchers who created it. YOLOv8 was released by the company Ultralytics in 2023 and is free to use.

What you will do:

1. Load the model.
2. Run it on a street photo.
3. Change how sure the model must be before it reports an object.
4. Try it on a photo of your own.
5. Test it on a construction site and in the dark.
6. Compare a small and a larger version of the model.
7. Look for a photo that fools it.

Each grey box below is a code cell. Click the play button on its left, wait until it finishes, and then read the output and the notes under it. You do not need to understand every line of code. Focus on what the model produces and what you make of it.

To keep your answers, choose File, then Save a copy in Drive, before you start.

## Step 1: Load the model

This cell installs the Ultralytics library and downloads YOLOv8 Nano, the smallest version of the model. It takes about a minute the first time.

The model was trained on COCO (Common Objects in Context), a collection of about 330,000 photos, more than 200,000 of them labeled by people, covering 80 kinds of everyday object. The cell prints all 80. People collected and labeled every one of those training photos, and the model can only recognize what those labels taught it.

In [ ]:
# Install the Ultralytics library (only needs to run once per session)
!pip install ultralytics --quiet

# Import the tools we need
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests
from io import BytesIO
import numpy as np

# Load YOLOv8 nano, the smallest and fastest version
# n = nano (fastest), s = small, m = medium, l = large, x = extra large
model = YOLO('yolov8n.pt')

print("Model loaded successfully")
print(f"This model can recognize {len(model.names)} kinds of object.")
print("\nEverything it knows about:")
for i, name in model.names.items():
    print(f"  {i:3d}: {name}")

## Step 2: Run the model on a street photo

The model draws a box around each object it finds and gives each box a confidence score between 0% and 100%. As you look at the result, notice what it gets right, what it misses, and whether a high score always means a correct answer.

In [ ]:
# Helper tools that run the model on a picture and show the result

def load_image(image_source):
    """Load an image from a URL, a file path, or a PIL image. Returns the image, or None if it fails."""
    try:
        if isinstance(image_source, str) and image_source.startswith('http'):
            resp = requests.get(image_source, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
            resp.raise_for_status()
            return Image.open(BytesIO(resp.content)).convert('RGB')
        elif isinstance(image_source, str):
            return Image.open(image_source).convert('RGB')
        else:
            return image_source.convert('RGB')
    except Exception as e:
        print("Could not load that image. The link may be down or blocked.")
        print("Tip: upload your own photo, or paste a different image URL.")
        print(f"(Details: {e})")
        return None


def show_detections(image_source, title="Object Detection", conf_threshold=0.25):
    """Run YOLOv8 on an image and show the boxes it draws."""
    img = load_image(image_source)
    if img is None:
        return []

    results = model(img, conf=conf_threshold, verbose=False)
    result = results[0]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(img); axes[0].set_title('Original Photo', fontsize=13, fontweight='bold'); axes[0].axis('off')
    axes[1].imshow(img); axes[1].set_title(title, fontsize=13, fontweight='bold'); axes[1].axis('off')

    colors = plt.cm.Set1(np.linspace(0, 1, 20))
    detections = []
    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0]); cls_id = int(box.cls[0])
            label = model.names[cls_id]; color = colors[cls_id % len(colors)]
            axes[1].add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                              linewidth=2.5, edgecolor=color, facecolor='none'))
            axes[1].text(x1, y1-5, f'{label} {conf:.0%}', color='white', fontsize=9, fontweight='bold',
                         bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85, edgecolor='none'))
            detections.append((label, conf))

    plt.suptitle(f'YOLOv8 found {len(detections)} objects (confidence at or above {conf_threshold:.0%})',
                 fontsize=12, y=1.01, color='#333')
    plt.tight_layout(); plt.show()

    print(f"\nWhat it found ({len(detections)} objects):")
    print("-" * 40)
    if detections:
        for label, conf in sorted(detections, key=lambda x: -x[1]):
            bar = '#' * int(conf * 20)
            print(f"  {label:20s} {conf:5.1%}  [{bar:<20}]")
    else:
        print("  Nothing found above the confidence threshold.")
    return detections


# Run it on a street scene
street_url = "https://ultralytics.com/images/bus.jpg"
detections = show_detections(street_url, title="Street Scene")

### Checkpoint 1: What did you see?

Double-click this cell to edit it, then write your answers.

**Your answers:**

1. What objects did the model get right?

   *Write here.*

2. Did it miss anything you can see in the photo?

   *Write here.*

3. Were there any confident guesses that surprised you?

   *Write here.*

## Step 3: Change the confidence threshold

The confidence threshold is how sure the model must be before it reports an object. With a high threshold, such as 0.75, only confident boxes appear. There are fewer mistakes, but real objects can be missed. With a low threshold, such as 0.05, shaky guesses appear too. More objects are found, but more of the boxes are wrong.

This balance is called the precision and recall trade-off. Precision is the share of the reported boxes that are correct. Recall is the share of the real objects that are found. If this model helped a car decide when to brake, would you set the threshold high or low? Why?

In [ ]:
# Change the number and run again. Try 0.05, 0.25, 0.5, 0.75, 0.9.
CONFIDENCE_THRESHOLD = 0.05   # <-- change this and run again

_ = show_detections(street_url,
                title=f"Confidence threshold: {CONFIDENCE_THRESHOLD:.0%}",
                conf_threshold=CONFIDENCE_THRESHOLD)

## Step 4: Try your own photo

Upload a photo from your phone or computer, for example your desk, a room or a street. Then run the cell after the upload cell to analyze it.

Challenge yourself to find a photo that confuses the model. Good candidates are objects seen from an odd angle, objects that are half hidden or somewhere unexpected, and photos that are blurry or dark.

In [ ]:
# Upload a photo from your computer or phone
from google.colab import files

print("Click the 'Choose Files' button below to upload a photo.")
print("Works with JPG, PNG, JPEG, and WEBP files.")

uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    print(f"\nUploaded: {filename}")
    print("Run the next cell to analyze it.")
else:
    print("No file uploaded yet.")

In [ ]:
# Analyze the photo you just uploaded (run the upload cell above first)
if 'uploaded' in globals() and uploaded:
    filename = list(uploaded.keys())[0]
    my_detections = show_detections(filename, title="Your Photo", conf_threshold=0.25)
else:
    print("Please run the upload cell above first.")

## Step 5: An engineering scene

YOLOv8 learned from everyday photos. Construction sites, circuit boards and factory floors look very different. Before you run the next cell, predict which objects the model will find on a construction site and which it will miss or get wrong.

In [ ]:
# A construction site (an engineering scene the model was not trained on)
# If this link is ever down, replace it with any construction or worksite photo URL.
construction_url = "https://images.unsplash.com/photo-1504307651254-35680f356dfd?w=800"

print("Before you run it: what do you think the model will detect here?")
print("Take 10 seconds to predict before you look.")
print()

eng_detections = show_detections(construction_url, title="Construction Site", conf_threshold=0.25)

## Step 6: Confidence is not the same as correct

A high confidence score does not mean the model is right. Low light is a good test of this. Most of the photos the model learned from were taken in good light; one study estimated that only about 0.2% of the images in COCO are low-light photos (Loh and Chan, 2019).

To test the effect of light alone, the next cell takes the street photo from Step 2 and makes it darker in three steps, adding a little camera noise each time, as a real night photo would have. Nothing else in the picture changes. Watch what happens to the number of boxes and to the confidence scores.

This connects to the face-analysis example in the opening demo. When some kinds of pictures are rare in the training data, a model tends to work less well on them.

In [ ]:
# Turn the street photo into a night photo: darker, with a little camera noise.
# Only the light changes, so any difference in the boxes comes from the darkness.
from PIL import ImageEnhance

street_photo = load_image(street_url)
noise = np.random.default_rng(0)

def make_night(img, brightness):
    """Darken a photo to the given brightness (1.0 = unchanged) and add camera noise."""
    dark = np.asarray(ImageEnhance.Brightness(img).enhance(brightness)).astype(float)
    dark = dark + noise.normal(0, 6, dark.shape)
    return Image.fromarray(np.clip(dark, 0, 255).astype('uint8'))

if street_photo is not None:
    for brightness in [1.0, 0.35, 0.2, 0.12]:
        _ = show_detections(make_night(street_photo, brightness),
                            title=f"Brightness {brightness:.0%}", conf_threshold=0.20)
    print()
    print("Compare the four results. What happened to the number of boxes and to the confidence scores?")
    print("What does that tell you about using AI in conditions it was not trained for?")

## Step 7: Compare two model sizes

YOLOv8 comes in five sizes, from nano (n) to extra large (x). Bigger models are usually more accurate, but they are slower and need more computing power. That matters in engineering, because a small chip inside a sensor has far less computing power than a server in a data center.

This cell runs the nano and small models on the same photo. Each model runs once to warm up and then three more times, and the cell reports the typical time.

In [ ]:
import time

# Load the small model (one step up from nano)
model_small = YOLO('yolov8s.pt')

# Use the street scene for the comparison
test_img = load_image(street_url)

# Run each model once first: the first run of any model is slow while it sets itself up
model(test_img, verbose=False)
model_small(test_img, verbose=False)

def time_model(m, runs=3):
    """Run a model several times and return its last result and its middle (median) time."""
    times = []
    for _ in range(runs):
        t0 = time.time()
        res = m(test_img, conf=0.25, verbose=False)
        times.append(time.time() - t0)
    return res, sorted(times)[runs // 2]

results_nano, t_nano = time_model(model)
results_small, t_small = time_model(model_small)

n_nano  = len(results_nano[0].boxes)  if results_nano[0].boxes  is not None else 0
n_small = len(results_small[0].boxes) if results_small[0].boxes is not None else 0

print("Model comparison: same photo, same threshold (25%)")
print("=" * 50)
print(f"  YOLOv8 Nano  (smallest): {n_nano:2d} objects  |  {t_nano*1000:.0f} ms")
print(f"  YOLOv8 Small (bigger):   {n_small:2d} objects  |  {t_small*1000:.0f} ms")
print()
print("For a real-time safety system running on a small chip,")
print("which would you choose, and what limits would drive that choice?")

## Step 8: Try to fool it

Find a photo where the model is confidently wrong. Some ideas to try:

- a dog in a costume
- a toy car next to a real shoe
- a photo turned on its side
- a close-up of a texture or pattern
- a drawing of an object instead of a photo of it

Paste an image link into the cell below, or upload another photo. Try at least two ideas.

In [ ]:
# Paste any image URL here and run it. The default below is the sample bus photo;
# replace it with something you think will fool the model.
test_url = "https://ultralytics.com/images/bus.jpg"

# Or upload another file instead: uncomment the two lines below.
# uploaded2 = files.upload()
# test_url = list(uploaded2.keys())[0]

_ = show_detections(test_url, title="Breaking Attempt", conf_threshold=0.20)

## Final reflection

Double-click this cell and write your answers.

1. What was the most surprising thing the model got right?

   *Write here.*

2. What was the most surprising thing the model got wrong, especially when it was confident?

   *Write here.*

3. The model learned from COCO, a collection of everyday photos of 80 kinds of object, very few of them taken in low light. Name one engineering job where this training data would make the model fail in a predictable way.

   *Write here.*

4. Imagine using object recognition on a construction site to spot workers, equipment and hazards. Name three things you would change about how the model was trained.

   *Write here.*

5. What is the difference between a model that is right 95% of the time in testing and a model that is safe to use on a real construction site?

   *Write here.*

---

Hands-On AI for High School · Dr. Ali Al Bataineh, Higher Colleges of Technology · aalbataineh@hct.ac.ae